This notebook is a basic tutorial that demonstrates how to configure a simulation using Concordia.

<a href="https://colab.research.google.com/github/google-deepmind/concordia/blob/main/examples/selling_cookies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title Colab-specific setup (use a CodeSpace to avoid the need for this).
try:
  %env COLAB_RELEASE_TAG
except:
  pass  # Not running in colab.
else:
  %pip install --ignore-requires-python --requirement 'https://raw.githubusercontent.com/google-deepmind/concordia/main/examples/requirements.in' 'git+https://github.com/google-deepmind/concordia.git#egg=gdm-concordia'
  %pip list

In [2]:
# @title Imports

import numpy as np
from IPython import display

import sentence_transformers

from concordia.language_model import utils as language_model_utils

from concordia.prefabs.simulation import generic as simulation

import concordia.prefabs.entity as entity_prefabs
import concordia.prefabs.game_master as game_master_prefabs

from concordia.typing import prefab as prefab_lib
from concordia.typing import scene as scene_lib
from collections.abc import Mapping, Sequence
from concordia.typing import entity as entity_lib

from concordia.utils import helper_functions

In [3]:
# @title Language Model Selection: provide key or select DISABLE_LANGUAGE_MODEL

# By default this colab uses models via an external API so you must provide an
# API key. TogetherAI offers open weights models from all sources.

API_KEY = 'blah'  #@param {type: 'string'}
# See concordia/language_model/utils.py
# API_TYPE = 'openai'  # e.g. 'together_ai' or 'openai'.
API_TYPE = 'ollama'
# MODEL_NAME = 'gpt-5'  # for API_TYPE = 'together_ai', we recommend MODEL_NAME = 'google/gemma-3-27b-it'
MODEL_NAME = 'deepseek-r1:32b'
# To debug without spending money on API calls, set DISABLE_LANGUAGE_MODEL=True
DISABLE_LANGUAGE_MODEL = False

In [4]:
# @title Use the selected language model

# Note that it is also possible to use local models or other API models,
# simply replace this cell with the correct initialization for the model
# you want to use.

if not DISABLE_LANGUAGE_MODEL and not API_KEY:
  raise ValueError('API_KEY is required.')

# model = language_model_utils.language_model_setup(
#     api_type=API_TYPE,
#     model_name=MODEL_NAME,
#     api_key=API_KEY,
#     disable_language_model=DISABLE_LANGUAGE_MODEL,
# )

model = language_model_utils.language_model_setup(
    api_type=API_TYPE,
    model_name=MODEL_NAME,
    disable_language_model=DISABLE_LANGUAGE_MODEL,
)

In [5]:
# @title Setup sentence encoder

if DISABLE_LANGUAGE_MODEL:
  embedder = lambda _: np.ones(3)
else:
  st_model = sentence_transformers.SentenceTransformer(
      'sentence-transformers/all-mpnet-base-v2')
  embedder = lambda x: st_model.encode(x, show_progress_bar=False)

In [6]:
# test = model.sample_text(
#     'Is societal and technological progress like getting a clearer picture of '
#     'something true and deep?')
# print(test)

In [7]:
# @title Load prefabs from packages to make the specific palette to use here.

prefabs = {
    **helper_functions.get_package_classes(entity_prefabs),
    **helper_functions.get_package_classes(game_master_prefabs),
}

In [8]:
#@title Print menu of prefabs

display.display(
    display.Markdown(helper_functions.print_pretty_prefabs(prefabs)))

---
**`basic__Entity`**:
```python
Entity(
    description='An entity that makes decisions by asking "What situation am I in right now?", "What kind of person am I?", and "What would a person like me do in a situation like this?"',
    params={'name': 'Alice', 'goal': '', 'randomize_choices': True}
)
```
---
**`basic_scripted__Entity`**:
```python
Entity(
    description='An entity that makes decisions by asking "What situation am I in right now?", "What kind of person am I?", and "What would a person like me do in a situation like this?"',
    params={'name': 'Alice', 'goal': '', 'script': []}
)
```
---
**`basic_with_plan__Entity`**:
```python
Entity(
    description='An entity that makes decisions by asking "What situation am I in right now?", "What kind of person am I?", and "What would a person like me do in a situation like this?" and building a plan based on the answers. It then tries to execute the plan.',
    params={'name': 'Alice', 'goal': '', 'force_time_horizon': False}
)
```
---
**`conversational__Entity`**:
```python
Entity(
    description='An entity that participates in conversations, aiming to create a dynamically balanced and engaging dialogue.',
    params={'name': 'Debra'}
)
```
---
**`fake_assistant_with_configurable_system_prompt__Entity`**:
```python
Entity(
    description='An entity that simulates an AI assistant with a configurable system prompt.',
    params={'name': 'Assistant', 'system_prompt': 'Assistant is a helpful and harmless AI assistant.'}
)
```
---
**`minimal__Entity`**:
```python
Entity(
    description='An entity that has a minimal set of components and is configurable by the user. The initial set of components manage memory, observations, and instructions. If goal is specified, the entity will have a goal constant component.',
    params={'name': 'Alice', 'goal': '', 'custom_instructions': '', 'extra_components': {}, 'extra_components_index': {}, 'randomize_choices': True}
)
```
---
**`dialogic__GameMaster`**:
```python
GameMaster(
    description='A game master specialized for handling conversation.',
    params={'name': 'conversation rules', 'next_game_master_name': 'default rules', 'acting_order': 'game_master_choice', 'can_terminate_simulation': True}
)
```
---
**`dialogic_and_dramaturgic__GameMaster`**:
```python
GameMaster(
    description='A game master specialized for handling conversation. This game master is designed to be used with scenes.',
    params={'name': 'conversation rules', 'scenes': ()}
)
```
---
**`formative_memories_initializer__GameMaster`**:
```python
GameMaster(
    description='An initializer for all entities that generates formative memories from their childhood.',
    params={'name': 'initial setup rules', 'next_game_master_name': 'default rules', 'shared_memories': [], 'player_specific_context': {}, 'player_specific_memories': {}}
)
```
---
**`game_theoretic_and_dramaturgic__GameMaster`**:
```python
GameMaster(
    description='A game master specialized for handling matrix game. decisions, designed to be used with scenes.',
    params={'name': 'decision rules', 'scenes': (), 'action_to_scores': <function _default_action_to_scores at 0x71b05aae37e0>, 'scores_to_observation': <function _default_scores_to_observation at 0x71b05aae3880>}
)
```
---
**`generic__GameMaster`**:
```python
GameMaster(
    description='A general purpose game master.',
    params={'name': 'default rules', 'extra_event_resolution_steps': '', 'extra_components': {}, 'extra_components_index': {}, 'acting_order': 'game_master_choice'}
)
```
---
**`interviewer__GameMaster`**:
```python
GameMaster(
    description='A game master that administers questionnaires to a specified player.',
    params={'name': 'InterviewerGM', 'player_names': [], 'questionnaires': [], 'verbose': False}
)
```
---
**`marketplace__GameMaster`**:
```python
GameMaster(
    description='A game master for marketplace simulations with inventory, positions, and transaction tracking.',
    params={'name': 'marketplace_gm', 'clock_description': 'The marketplace operates during daytime hours. Time passes with each action, movement, and conversation. Each movement between positions takes 1 time unit. Each round of conversation takes 1 time unit.', 'start_time': '9:00 AM', 'locations': '', 'extra_event_resolution_steps': '', 'item_types': [], 'player_initial_endowments': {}, 'financial': True, 'inventory_verbose': False}
)
```
---
**`open_ended_interviewer__GameMaster`**:
```python
GameMaster(
    description='A game master that administers questionnaires to a specified player.',
    params={'name': 'InterviewerGM', 'player_names': [], 'questionnaires': [], 'sequence_of_events': [], 'embedder': None, 'verbose': False}
)
```
---
**`psychology_experiment__GameMaster`**:
```python
GameMaster(
    description='A generic Game Master that administers a psychology experiment defined by custom observation and action specification components.',
    params={'name': 'ExperimenterGM', 'scenes': (), 'experiment_component_class': None, 'experiment_component_init_kwargs': {}}
)
```
---
**`scripted__GameMaster`**:
```python
GameMaster(
    description='A game master that administers questionnaires to a specified player.',
    params={'name': 'ScriptedGM', 'script': [], 'verbose': False}
)
```
---
**`situated__GameMaster`**:
```python
GameMaster(
    description='A general game master for games set in a specific location.',
    params={'name': 'default rules', 'extra_event_resolution_steps': '', 'locations': '', 'extra_components': {}, 'extra_components_index': {}}
)
```
---
**`situated_in_time_and_place__GameMaster`**:
```python
GameMaster(
    description='A general game master for games situated in a physical time/place.',
    params={'name': 'default rules', 'extra_event_resolution_steps': '', 'clock_description': "The passing of time can be conveyed using any convenient feature of the environment, e.g. a physical clock, the angle of the sun, extent of a candle's melting, phase of the moon, agricultural season, elapsed time since an event, etc. Whenever possible, try to track the day and year as well as the time within the day. To determine the passing of time, try to make reasonable inferences about the amount of time that would most likely have elapsed between the previous event and the latest event, taking into account the number of simulation steps taken.", 'start_time': '', 'locations': '', 'extra_components': {}, 'extra_components_index': {}}
)
```
---
**`stpgm_playground_copy__GameMaster`**:
```python
GameMaster(
    description='A general game master for games situated in a physical time/place.',
    params={'name': 'default rules', 'extra_event_resolution_steps': '', 'clock_description': "The passing of time can be conveyed using any convenient feature of the environment, e.g. a physical clock, the angle of the sun, extent of a candle's melting, phase of the moon, agricultural season, elapsed time since an event, etc. Whenever possible, try to track the day and year as well as the time within the day. To determine the passing of time, try to make reasonable inferences about the amount of time that would most likely have elapsed between the previous event and the latest event, taking into account the number of simulation steps taken.", 'start_time': '', 'locations': '', 'inventory_item_type_configs': [], 'inventory_initial_endowments': {}, 'extra_components': {}, 'extra_components_index': {}}
)
```
---

In [9]:
"""A prefab implementing an entity with a minimal set of components."""

from collections.abc import Mapping
import dataclasses

from concordia.agents import entity_agent_with_logging
from concordia.associative_memory import basic_associative_memory
from concordia.components import agent as agent_components
from concordia.language_model import language_model
from concordia.typing import prefab as prefab_lib

DEFAULT_INSTRUCTIONS_COMPONENT_KEY = 'Instructions'
DEFAULT_INSTRUCTIONS_PRE_ACT_LABEL = '\nInstructions'
DEFAULT_GOAL_COMPONENT_KEY = 'Goal'


@dataclasses.dataclass
class MyAgent(prefab_lib.Prefab):
  """A prefab implementing an entity with a minimal set of components."""

  description: str = (
      'An entity that has a minimal set of components and is configurable by'
      ' the user. The initial set of components manage memory, observations,'
      ' and instructions. If goal is specified, the entity will have a goal '
      'constant component.'
  )
  params: Mapping[str, str] = dataclasses.field(
      default_factory=lambda: {
          'name': 'Alice',
          'goal': '',
      }
  )

  def build(
      self,
      model: language_model.LanguageModel,
      memory_bank: basic_associative_memory.AssociativeMemoryBank,
  ) -> entity_agent_with_logging.EntityAgentWithLogging:
    """Build an agent.

    Args:
      model: The language model to use.
      memory_bank: The agent's memory_bank object.

    Returns:
      An entity.
    """

    agent_name = self.params.get('name', 'Alice')

    instructions = agent_components.instructions.Instructions(
          agent_name=agent_name,
          pre_act_label=DEFAULT_INSTRUCTIONS_PRE_ACT_LABEL,
      )

    observation_to_memory = agent_components.observation.ObservationToMemory()

    observation_label = '\nObservation'
    observation = agent_components.observation.LastNObservations(
        history_length=100, pre_act_label=observation_label
    )

    principle = agent_components.question_of_recent_memories.QuestionOfRecentMemories(
        model=model,
        pre_act_label=f'{agent_name} main guiding principle:',
        question=(f'How can {agent_name} exploit the situation for personal '
                  'gain and gratification?'),
        answer_prefix=f'{agent_name} understands that ',
        add_to_memory=False,
    )

    components_of_agent = {
        DEFAULT_INSTRUCTIONS_COMPONENT_KEY: instructions,
        'observation_to_memory': observation_to_memory,
        agent_components.observation.DEFAULT_OBSERVATION_COMPONENT_KEY: (
            observation
        ),
        agent_components.memory.DEFAULT_MEMORY_COMPONENT_KEY: (
            agent_components.memory.AssociativeMemory(memory_bank=memory_bank)
        ),
        'principle': principle,
    }

    component_order = list(components_of_agent.keys())

    if self.params.get('goal', ''):
      goal_key = DEFAULT_GOAL_COMPONENT_KEY
      goal = agent_components.constant.Constant(
          state=self.params.get('goal', ''),
          pre_act_label='Overarching goal',
      )
      components_of_agent[goal_key] = goal
      # Place goal after the instructions.
      component_order.insert(1, goal_key)

    act_component = agent_components.concat_act_component.ConcatActComponent(
        model=model,
        component_order=component_order,
    )

    agent = entity_agent_with_logging.EntityAgentWithLogging(
        agent_name=agent_name,
        act_component=act_component,
        context_components=components_of_agent,
    )

    return agent


In [10]:
prefabs['myagent__Entity'] = MyAgent()

## ORIGINAL SETUP

In [11]:
# DEFAULT_NAME = 'decision rules'

# PLAYER_ONE = 'Alice'
# PLAYER_TWO = 'Bob'

# def configure_scenes() -> Sequence[scene_lib.SceneSpec]:
#   """Configure default scenes for this simulation."""
#   decision = scene_lib.SceneTypeSpec(
#       name='decision',
#       game_master_name=DEFAULT_NAME,
#       action_spec = {
#           PLAYER_ONE: entity_lib.choice_action_spec(
#               call_to_action='Would {name} buy the cookies from Bob?',
#               options=['Yes', 'No'],
#           ),
#       }
#   )

#   check_inventory = scene_lib.SceneTypeSpec(
#       name='check_inventory',
#       game_master_name='stpinventory rules',
#       action_spec=entity_lib.free_action_spec(call_to_action="What does {name} observe about their inventory?"),
#   )

#   conversation = scene_lib.SceneTypeSpec(
#       name='conversation',
#       game_master_name='conversation rules',
#       action_spec=entity_lib.free_action_spec(call_to_action=entity_lib.DEFAULT_CALL_TO_SPEECH),
#       )

#   scenes = [
#       scene_lib.SceneSpec(
#           scene_type=conversation,
#           participants=[PLAYER_ONE, PLAYER_TWO],
#           num_rounds=4,
#           premise={
#               PLAYER_ONE : [f'{PLAYER_ONE} is approached by {PLAYER_TWO}'],
#               PLAYER_TWO : [f'{PLAYER_TWO} has approached {PLAYER_ONE}'],
#           },
#           ),
#       scene_lib.SceneSpec(
#           scene_type=check_inventory,  # Add your inventory scene here
#           participants=[PLAYER_ONE, PLAYER_TWO],
#           num_rounds=1,
#           premise={
#               PLAYER_ONE: [f'{PLAYER_ONE} checks their belongings'],
#               PLAYER_TWO: [f'{PLAYER_TWO} checks their belongings'],
#           },
#       ),
#       scene_lib.SceneSpec(
#           scene_type=decision,
#           participants=[PLAYER_ONE],
#           num_rounds=1,
#           premise={
#               PLAYER_ONE : [f'{PLAYER_ONE} has to decide whether to buy cookies from {PLAYER_TWO}'],
#           },
#       ),
#   ]
#   return scenes

# def action_to_scores(
#     joint_action: Mapping[str, str],
# ) -> Mapping[str, float]:
#   """Map a joint action to a dictionary of scores for each player."""
#   if joint_action[PLAYER_ONE] == 'Yes':
#     return {PLAYER_ONE: -1, PLAYER_TWO: 1}
#   return  {PLAYER_ONE: 1, PLAYER_TWO: -1}


# def scores_to_observation(
#     scores: Mapping[str, float]) -> Mapping[str, str]:
#   """Map a dictionary of scores for each player to a string observation.

#   This function is appropriate for a coordination game structure.

#   Args:
#     scores: A dictionary of scores for each player.

#   Returns:
#     A dictionary of observations for each player.
#   """
#   observations = {}
#   for player_name in scores:
#     if scores[player_name] > 0:
#       observations[player_name] = (
#           f'{player_name} enjoyed the transaction.'
#       )
#     else:
#       observations[player_name] = (
#           f'{player_name} did not enjoy the transaction.'
#       )
#   return observations


In [12]:
# scenes = configure_scenes()

In [13]:
# # @title Configure instances.
# from types import SimpleNamespace

# instances = [
#     prefab_lib.InstanceConfig(
#         prefab='basic__Entity',
#         role=prefab_lib.Role.ENTITY,
#         params={
#             'name': PLAYER_ONE,
#         },
#     ),
#     prefab_lib.InstanceConfig(
#         prefab='myagent__Entity',
#         role=prefab_lib.Role.ENTITY,
#         params={
#             'name': PLAYER_TWO,
#             'goal': f'Sell cookies to {PLAYER_ONE}',
#         },
#     ),
#     prefab_lib.InstanceConfig(
#         prefab='game_theoretic_and_dramaturgic__GameMaster',
#         role=prefab_lib.Role.GAME_MASTER,
#         params={
#             'name': 'decision rules',
#             # Comma-separated list of thought chain steps.
#             'scenes': scenes,
#             'action_to_scores': action_to_scores,
#             'scores_to_observation': scores_to_observation,
#         },
#     ),
#     prefab_lib.InstanceConfig(
#         prefab='stpgm_playground_copy__GameMaster',
#         role=prefab_lib.Role.GAME_MASTER,
#         params={
#           'name': 'stpinventory rules',
#           'extra_event_resolution_steps': '',
#           'clock_description': "Whenever possible, try to track the day and year as well as the time within the day. To determine the passing of time, try to make reasonable inferences about the amount of time that would most likely have elapsed between the previous event and the latest event, taking into account the number of simulation steps taken.",
#           'start_time': '2025-12-02 01:39:00',
#           'locations': 'porch',
#           'inventory_item_type_configs': [SimpleNamespace(name="dollars"), SimpleNamespace(name="cookies")],
#           'inventory_initial_endowments': {"Alice": {"dollars": 10.0}, "Bob": {"cookies": 5.0}},
#           'extra_components': {},
#           'extra_components_index': {},
#           'verbose': True,
#         },
#     ),
#     prefab_lib.InstanceConfig(
#         prefab='dialogic_and_dramaturgic__GameMaster',
#         role=prefab_lib.Role.GAME_MASTER,
#         params={
#             'name': 'conversation rules',
#             # Comma-separated list of thought chain steps.
#             'scenes': scenes,
#         },
#     ),
#     prefab_lib.InstanceConfig(
#         prefab='formative_memories_initializer__GameMaster',
#         role=prefab_lib.Role.INITIALIZER,
#         params={
#             'name': 'initial setup rules',
#             'next_game_master_name': 'conversation rules',
#             'shared_memories': [
#                 f'There is a small town of Riverbend where {PLAYER_ONE} and {PLAYER_TWO} grew up.',
#             ],
#             'player_specific_memories': {PLAYER_ONE : [f'{PLAYER_ONE} will do anything for a charitable cause.',
#                                                        f'{PLAYER_ONE} does not like cookies'],
#                                          PLAYER_TWO : [f'{PLAYER_TWO} is a cookie salesman.']},
#             'player_specific_context': {PLAYER_ONE : f'{PLAYER_ONE} does not like cookies.',
#                                          PLAYER_TWO : f'{PLAYER_TWO} is a cookie salesman.'},
#         },
#     ),
# ]

In [14]:
# config = prefab_lib.Config(
#     default_premise=(
#         'It is a bright sunny day in the town of Riverbend. The sun is in the'
#         f' zenith and the gentle breeze is rocking the trees. {PLAYER_ONE} is'
#         f' standing on the porch of their house. {PLAYER_TWO} has approached'
#         f' {PLAYER_ONE}'
#     ),
#     default_max_steps=5,
#     prefabs=prefabs,
#     instances=instances,
# )

## MY SETUP TO TEST MY CUSTOM GM (SITUATED TIME PLACE WITH INVENTORY, NON-SCENE BASED)

In [ ]:
# @title Simple Inventory Test Configuration

from types import SimpleNamespace
from concordia.components.game_master import inventory


DEFAULT_NAME = 'inventory rules'
PLAYER_ONE = 'Alice'
PLAYER_TWO = 'Bob'

# Configure instances - much simpler than the scene-based version!
instances = [
    # Alice - a basic entity
    prefab_lib.InstanceConfig(
        prefab='basic__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': PLAYER_ONE,
        },
    ),

    # Bob - your custom agent with the exploitation principle
    prefab_lib.InstanceConfig(
        prefab='myagent__Entity',  # The custom agent you created in the notebook
        role=prefab_lib.Role.ENTITY,
        params={
            'name': PLAYER_TWO,
            'goal': f'Sell cookies to {PLAYER_ONE}',
        },
    ),

    # Your inventory-enabled game master (non-scene-based)
    prefab_lib.InstanceConfig(
        prefab='stpgm_playground_copy__GameMaster',  # Replace with your actual prefab name
        role=prefab_lib.Role.GAME_MASTER,
        params={
            'name': DEFAULT_NAME,
            'clock_description': (
                'The marketplace operates during daytime hours. Time passes with each '
                'action and conversation. It is currently a sunny afternoon.'
            ),
            'start_time': '2:00 PM',
            'locations': (
                'A small marketplace in Riverbend. There is a cookie stall where Bob '
                'works.'
            ),
            # 'inventory_item_type_configs': [
            #     {'name': 'cookies', 'minimum': 0},
            #     {'name': 'money', 'minimum': 0},
            # ],
            # 'inventory_initial_endowments': {
            #     PLAYER_ONE: {'money': 10, 'cookies': 0},
            #     PLAYER_TWO: {'cookies': 12, 'money': 0},
            # },
            # 'inventory_item_type_configs': [SimpleNamespace(name="money", minimum=0), SimpleNamespace(name="cookies", minimum=0)],
            'inventory_item_type_configs': [
                inventory.ItemTypeConfig(name='money', minimum=0),
                inventory.ItemTypeConfig(name='cookies', minimum=0),
            ],
            'inventory_initial_endowments': {"Alice": {"money": 10.0}, "Bob": {"cookies": 5.0}},
        },
    ),

    # Initializer to set up memories
    prefab_lib.InstanceConfig(
        prefab='formative_memories_initializer__GameMaster',
        role=prefab_lib.Role.INITIALIZER,
        params={
            'name': 'initial setup rules',
            'next_game_master_name': DEFAULT_NAME,  # Hand off to your inventory GM
            'shared_memories': [
                f'There is a small marketplace in Riverbend where {PLAYER_ONE} and {PLAYER_TWO} often meet.',
                f'{PLAYER_TWO} runs a cookie stall in the marketplace. Each cookie costs one dollar.',
            ],
            'player_specific_memories': {
                PLAYER_ONE: [
                    f'{PLAYER_ONE} has some money to spend today.',
                    f'{PLAYER_ONE} likes cookies and is considering buying three cookies.',
                ],
                PLAYER_TWO: [
                    f'{PLAYER_TWO} is a cookie salesman.',
                    f'{PLAYER_TWO} needs to sell cookies to make a living.',
                ],
            },
            'player_specific_context': {
                PLAYER_ONE: f'{PLAYER_ONE} has money and is walking through the marketplace.',
                PLAYER_TWO: f'{PLAYER_TWO} is at their cookie stall hoping to make a sale.',
            },
        },
    ),
]

In [16]:
# Create the config
config = prefab_lib.Config(
    default_premise=(
        f'It is a sunny afternoon in the Riverbend marketplace. {PLAYER_ONE} is '
        f'walking past the stalls, and {PLAYER_TWO} is standing at their cookie '
        f'stall, arranging fresh-baked goods on display.'
    ),
    default_max_steps=3,  # Run for 10 steps to see interactions
    prefabs=prefabs,
    instances=instances,
)

# The simulation

In [17]:
# @title Initialize the simulation
runnable_simulation = simulation.Simulation(
    config=config,
    model=model,
    embedder=embedder,
)

In [18]:
# @title Run the simulation
raw_log = []
results_log = runnable_simulation.play(max_steps=3,
                                       raw_log=raw_log)

Terminate? No
Game master: initial setup rules
Entity Alice observed: There is a small marketplace in Riverbend where Alice and Bob often meet.


Bob runs a cookie stall in the marketplace. Each cookie costs one dollar.


When Alice was 25 years old, she took on her first major case as a social worker. She helped a single mother struggling to provide for her children, finding them food and shelter while also advocating for better resources in their community. This experience solidified her commitment to fighting systemic inequities.






At age 30, Alice faced adversity when a family she was helping was threatened with eviction. Despite initial setbacks, she campaigned tirelessly, rallying support from the town to save their home. This taught her the power of collective action and deepened her resolve to create lasting change.






When Alice turned 40, she began mentoring new social workers, sharing her experiences and strategies for navigating tough cases. Through teaching others, 

In [19]:
# @title Display the log
display.HTML(results_log)

```
Copyright 2024 DeepMind Technologies Limited.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
```